# Shared enums - Rust

All 21 Rust examples from [docs/enums.md](https://platob.github.io/yggdryl/enums/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::{DataType, DataTypeId, DataTypeKind};

let value = DataType::from_str("int64")?;
assert_eq!(value.id(), DataTypeId::Int64);
assert_eq!(value.kind(), DataTypeKind::Integer);
assert_eq!(value.id().as_str(), "int64");

## Identity carries no parameters

In [ ]:
use yggdryl::{DataType, DataTypeId};

let stamp = DataType::from_str("timestamp(us, UTC)")?;
assert_eq!(stamp.id(), DataTypeId::Timestamp);
assert_eq!(stamp.to_string(), "timestamp(us,\"UTC\")");
assert!(DataTypeId::Timestamp.is_parameterized());
assert!(!DataTypeId::Int32.is_parameterized());

In [ ]:
use yggdryl::{DataTypeId, DataTypeKind};

assert_eq!(DataTypeId::ALL.len(), 45);
assert_eq!(DataTypeKind::ALL.len(), 16);

assert_eq!(DataTypeId::Int32.fixed_byte_width(), Some(4));
assert_eq!(DataTypeId::Utf8.fixed_byte_width(), None);
assert!(DataTypeId::Int32.is_signed_integer() && !DataTypeId::Int32.is_unsigned_integer());

// A wrapper is nested only when the value it encodes is, so it reports neither.
assert!(DataTypeKind::Dictionary.is_wrapper());
assert!(!DataTypeKind::Dictionary.is_nested());
assert!(DataTypeKind::Struct.is_nested());

## MIME types

In [ ]:
use yggdryl::MimeType;

let parquet = MimeType::from_extension("parquet")?;
assert_eq!(parquet, MimeType::PARQUET);
assert_eq!(parquet.as_str(), "application/vnd.apache.parquet");
assert_eq!(parquet.top_level(), "application");
assert!(parquet.is_tabular() && parquet.is_binary());

let custom = MimeType::from_str("Application/Vnd.Example+JSON")?;
assert_eq!(custom.as_str(), "application/vnd.example+json");
assert_eq!(custom.structured_suffix(), Some("json"));
assert!(!custom.is_known());
assert!(custom.is_structured());

In [ ]:
use yggdryl::MimeType;

assert_eq!(
    MimeType::from_content_type("Application/JSON; charset=\"utf-8\"")?,
    MimeType::JSON
);
assert!(MimeType::from_content_type("application/json; charset").is_err());

// Content codings map both directions, and `identity` is not one of them.
assert_eq!(MimeType::from_content_coding("x-gzip")?, MimeType::GZIP);
assert_eq!(MimeType::GZIP.content_coding(), Some("gzip"));
assert!(MimeType::from_content_coding("identity").is_err());

## Media types are a base plus its codings

In [ ]:
use yggdryl::{MediaType, MimeType};

let media = MediaType::from_file_name("trades.json.gz");
assert_eq!(media.base(), &MimeType::JSON);
assert_eq!(media.encodings(), &[MimeType::GZIP]);
assert_eq!(media.encoding(), Some(&MimeType::GZIP));
assert_eq!(media.extensions().collect::<Vec<_>>(), ["json", "gz"]);
assert!(media.is_encoded());
assert_eq!(media.to_string(), "application/json;encodings=application/gzip");

In [ ]:
use yggdryl::{MediaType, MimeType};

let media = MediaType::from_content_headers(Some("text/csv; charset=utf-8"), Some("gzip"))?;
assert_eq!(media.base(), &MimeType::CSV);
assert_eq!(media.encoding(), Some(&MimeType::GZIP));

// Compound suffixes name both halves at once.
assert_eq!(
    MediaType::from_extension("tgz"),
    MediaType::from_parts(MimeType::TAR, [MimeType::GZIP])?
);

// Only a coding may be pushed; anything else leaves the value untouched.
let mut stacked = MediaType::from_file_name("events.json");
assert!(stacked.push_encoding(MimeType::ZIP).is_err());
stacked.push_encoding(MimeType::ZSTD)?;
assert_eq!(stacked.extension(), Some("zst"));

## Directories and files

In [ ]:
use yggdryl::MimeType;

assert_eq!(MimeType::DIRECTORY.as_str(), "inode/directory");
assert_eq!(MimeType::FILE.as_str(), "inode/file");
assert!(MimeType::DIRECTORY.is_directory());
assert!(MimeType::FILE.is_filesystem() && !MimeType::FILE.is_directory());
assert!(!MimeType::DIRECTORY.is_io());
assert!(MimeType::FILE.is_io() && MimeType::CSV.is_io());
assert_eq!(MimeType::DIRECTORY.extension(), None);

let directory = std::env::temp_dir();
assert_eq!(MimeType::from_local_path(&directory), MimeType::DIRECTORY);
assert_eq!(MimeType::from_local_path(directory.join("report.csv")), MimeType::CSV);
// A name that says nothing is still known to be a leaf.
assert_eq!(MimeType::from_local_path(directory.join("payload")), MimeType::FILE);

## Content inference from bytes

In [ ]:
use yggdryl::enums::MAGIC_PROBE_LEN;
use yggdryl::{MediaType, MimeType, gzip, zstd};

assert_eq!(MimeType::from_magic_bytes(b"PAR1"), Some(MimeType::PARQUET));
assert_eq!(MimeType::from_magic_bytes(b"ARROW1\0\0"), Some(MimeType::ARROW_FILE));

// Text has no signature, so it is sniffed structurally and only when unambiguous.
assert_eq!(MimeType::from_bytes(b"  {\"symbol\": \"AAPL\"}"), Some(MimeType::JSON));
assert_eq!(MimeType::from_bytes(b"key = 1"), None);

// Codings are peeled recursively and reported in application order.
let payload = gzip::dump(&zstd::dump(br#"{"symbol":"AAPL"}"#)?)?;
let media = MediaType::from_magic_bytes(&payload).expect("gzip of zstd of json");
assert_eq!(media.base(), &MimeType::JSON);
assert_eq!(media.encodings(), &[MimeType::GZIP, MimeType::ZSTD]);

assert_eq!(MAGIC_PROBE_LEN, 64);

## Schemes

In [ ]:
use yggdryl::Scheme;

assert_eq!(Scheme::HTTPS.default_port(), Some(443));
assert_eq!(Scheme::POSTGRES.default_port(), Some(5432));
assert_eq!(Scheme::S3.default_port(), None);
assert!(Scheme::S3.is_storage() && !Scheme::ICEBERG.is_storage());

assert_eq!(Scheme::COMPATIBILITY_TARGETS.len(), 5);
assert!(Scheme::SPARK.is_compatibility_target());
assert!(Scheme::ICEBERG.is_compatibility_target());
assert!(!Scheme::HTTPS.is_compatibility_target());

// Any RFC-valid scheme parses; only the listed ones are allocation-free.
let custom = Scheme::from_str("Acme+Wire")?;
assert_eq!(custom.as_str(), "acme+wire");
assert!(!custom.is_known() && Scheme::HTTPS.is_known());

## Content codings

In [ ]:
use yggdryl::{Codec, Level, Url};

let plain = b"symbol,price\nAAPL,1\n";
let compressed = Codec::Gzip.dump_with_level(plain, Level::BEST)?;
assert_eq!(Codec::Gzip.load(&compressed)?, plain.to_vec());
assert_eq!(Codec::Identity.dump(plain)?, plain.to_vec());

// The coding is recoverable from a filename alone.
let url = Url::from_str("file:///trades.csv.gz")?;
assert_eq!(Codec::from_url(&url), Codec::Gzip);
assert_eq!(Codec::from_mime_type(&yggdryl::MimeType::ZSTD), Codec::Zstd);
assert_eq!(Codec::Gzip.extension(), Some("gz"));
assert!(Codec::Identity.is_identity());

In [ ]:
use yggdryl::Level;

assert_eq!(Level::NONE.get(), 0);
assert_eq!(Level::DEFAULT.get(), 6);
assert_eq!(Level::BEST.get(), 9);
assert_eq!(Level::default(), Level::DEFAULT);
// Out-of-range levels clamp rather than fail.
assert_eq!(Level::new(200), Level::BEST);

## What a handle addresses

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::IOKind;

assert_eq!(Buffer::new().kind(), IOKind::Memory);
assert_eq!(IOKind::from_str("directory")?, IOKind::Directory);
assert_eq!(IOKind::default(), IOKind::File);

assert!(IOKind::Memory.is_leaf() && IOKind::File.is_leaf());
assert!(IOKind::Directory.is_container());
assert!(!IOKind::Unknown.is_known());

In [ ]:
use yggdryl::IOKind;

for container in [IOKind::Table, IOKind::Namespace, IOKind::Catalog] {
    assert!(container.is_container());
    assert!(!container.is_leaf());
}
assert_eq!(IOKind::from_str("CATALOG")?, IOKind::Catalog);
// The parser names the whole vocabulary when it refuses one.
let refused = IOKind::from_str("warehouse").unwrap_err().to_string();
assert!(refused.contains("namespace"));

## Record write intent

In [ ]:
use yggdryl::WriteMode;

assert_eq!(WriteMode::from_str("OVERWRITE")?, WriteMode::Overwrite);
assert_eq!(WriteMode::Append.as_str(), "append");
assert_eq!(WriteMode::Merge.to_string(), "merge");
assert_eq!(
    WriteMode::ALL.map(WriteMode::as_str),
    ["overwrite", "append", "merge"],
);

## Time units and union modes

In [ ]:
use yggdryl::{DataType, TimeUnit};

assert_eq!(TimeUnit::from_str("microseconds")?, TimeUnit::Microsecond);
assert_eq!(TimeUnit::from_str("MICRO SECONDS")?, TimeUnit::Microsecond);
assert_eq!(TimeUnit::Microsecond.as_str(), "us");

// The unit picks the physical width.
assert_eq!(DataType::time(TimeUnit::Second)?.to_string(), "time32(s)");
assert_eq!(DataType::time(TimeUnit::Microsecond)?.to_string(), "time64(us)");

In [ ]:
use yggdryl::TimeUnit;

assert!(TimeUnit::Nanosecond.is_temporal() && !TimeUnit::Nanosecond.is_interval());
assert!(TimeUnit::MonthDayNano.is_interval());
assert_eq!(TimeUnit::MonthDayNano.as_str(), "month_day_nano");

assert_eq!(TimeUnit::Nanosecond.into_arrow_time()?, arrow_schema::TimeUnit::Nanosecond);
assert!(TimeUnit::MonthDayNano.into_arrow_time().is_err());
assert!(TimeUnit::Nanosecond.into_arrow_interval().is_err());

In [ ]:
use yggdryl::UnionMode;

assert_eq!(UnionMode::Sparse.as_str(), "sparse");
assert_eq!(UnionMode::Dense.to_string(), "dense");
assert_ne!(UnionMode::Sparse, UnionMode::Dense);

## Edge algorithms

In [ ]:
use yggdryl::{DataType, EdgeAlgorithm};

assert_eq!(EdgeAlgorithm::ALL.len(), 5);
assert_eq!(EdgeAlgorithm::default(), EdgeAlgorithm::Spherical);
assert_eq!(EdgeAlgorithm::Vincenty.as_str(), "vincenty");

// Parsing is ASCII case-insensitive; display is the canonical lowercase name.
assert_eq!(EdgeAlgorithm::from_str("KARNEY")?, EdgeAlgorithm::Karney);
assert_eq!(EdgeAlgorithm::Andoyer.to_string(), "andoyer");

// An unknown name reports the input and the whole accepted vocabulary.
let error = EdgeAlgorithm::from_str("euclidean").unwrap_err();
assert!(error.to_string().contains("expected one of spherical"));

// The value lives on a geography datatype and nowhere else.
let vincenty = DataType::geography(None, Some(EdgeAlgorithm::Vincenty))?;
assert_eq!(vincenty.to_string(), "geography(\"OGC:CRS84\",\"vincenty\")");

## Listing the vocabularies

In [ ]:
use yggdryl::{Codec, DataTypeId, IOKind, TimeUnit, UnionMode};

// Every core enum publishes its variants in canonical order. Check
// representatives rather than pinning an extensible vocabulary's length.
assert!(DataTypeId::ALL.contains(&DataTypeId::Int64));
assert!(DataTypeId::ALL.contains(&DataTypeId::Struct));
assert!(DataTypeId::ALL.contains(&DataTypeId::Geography));
assert_eq!(UnionMode::ALL.map(UnionMode::as_str), ["sparse", "dense"]);
assert!(TimeUnit::ALL.contains(&TimeUnit::Microsecond));
assert!(Codec::ALL.contains(&Codec::Gzip));
assert!(IOKind::ALL.contains(&IOKind::File));

## Timezone

In [ ]:
use yggdryl::Timezone;

// Two spellings of one zone are one value.
assert_eq!(Timezone::from_str("Asia/Calcutta")?, Timezone::from_str("Asia/Kolkata")?);
assert_eq!(Timezone::from_str("US/Eastern")?.as_str(), "America/New_York");
assert_eq!(Timezone::from_str("Z")?, Timezone::UTC);
assert_eq!(Timezone::from_str("+0530")?.as_str(), "+05:30");

// A registered zone knows the rule in force today.
let new_york = Timezone::from_str("America/New_York")?;
assert_eq!(new_york.offset_at(1_705_000_000), Some(-5 * 3600));
assert_eq!(new_york.offset_at(1_720_000_000), Some(-4 * 3600));
assert_eq!(new_york.abbreviation_at(1_720_000_000), Some("EDT"));
assert_eq!(new_york.standard_offset(), Some(-5 * 3600));

// A zone with no known rules answers nothing rather than guessing.
let custom = Timezone::from_str("Custom/Accepted")?;
assert!(!custom.is_known());
assert_eq!(custom.offset_at(0), None);